# BERT Fine-Tuning on Yelp Reviews (English)

**Run this notebook on Google Colab** (Runtime → Change runtime type → T4 GPU)

Fine-tunes `bert-base-uncased` on the Yelp Review Full dataset (5-star → 3-class).
Uses 10K training samples for fast iteration.

**Important**: Test set sampling uses `RANDOM_SEED=42` with stratified split to ensure
the exact same 5K test samples are used across all model evaluations (Classical ML, BERT, LLM).
This is critical for a fair comparison.

**Label mapping rationale**: We map 1-2★ → negative, 3★ → neutral, 4-5★ → positive,
following standard practice (Zhang et al., 2015). Trade-off: 3★ is only ~20% of data,
creating class imbalance. We chose 3-class for consistency with the German Sentiment dataset.

Outputs:
- Trained model saved to Google Drive (or downloaded)
- Metrics JSON for comparison with other models

In [ ]:
# Install dependencies
!pip install -q datasets accelerate scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 1. Load Yelp Dataset

Yelp Review Full has 650K train / 50K test with 5-star labels.
We map to 3 classes: **negative** (1-2★), **neutral** (3★), **positive** (4-5★).

**Sampling strategy**:
- 10K train samples (stratified) — sufficient for BERT fine-tuning
- 5K test samples (stratified, RANDOM_SEED=42) — **identical** to what `loader.py` produces
  with `max_test_samples=5000`, ensuring cross-model comparability

Note: The test set subsampling always uses `RANDOM_SEED=42` regardless of training seed,
so that all models are evaluated on the exact same test data.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_LABELS = 3
RANDOM_SEED = 42

# 5-star (0-4 in HF) -> 3-class mapping
STAR_TO_SENTIMENT = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}

# Load dataset
ds = load_dataset("Yelp/yelp_review_full")

# Convert to pandas for easy manipulation
train_full = ds["train"].to_pandas()
test_full = ds["test"].to_pandas()

# Map labels
train_full["label"] = train_full["label"].map(STAR_TO_SENTIMENT)
test_full["label"] = test_full["label"].map(STAR_TO_SENTIMENT)

# Sample 10K for training (stratified)
MAX_TRAIN = 10000
train_df, _ = train_test_split(
    train_full, train_size=MAX_TRAIN, random_state=RANDOM_SEED,
    stratify=train_full["label"]
)

# Split into train + val (85/15)
train_split, val_split = train_test_split(
    train_df, test_size=0.15, random_state=RANDOM_SEED,
    stratify=train_df["label"]
)

# Use 5K test samples (stratified)
MAX_TEST = 5000
test_df, _ = train_test_split(
    test_full, train_size=MAX_TEST, random_state=RANDOM_SEED,
    stratify=test_full["label"]
)

print(f"Train: {len(train_split)} samples")
print(f"Val:   {len(val_split)} samples")
print(f"Test:  {len(test_df)} samples")
print(f"\nLabel distribution (train):")
print(train_split["label"].value_counts().sort_index().rename(index={0: 'negative', 1: 'neutral', 2: 'positive'}))

## 2. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )

# Convert pandas back to HF Dataset for Trainer
train_ds = Dataset.from_pandas(train_split[["text", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_split[["text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize_fn, batched=True, batch_size=256)
val_ds = val_ds.map(tokenize_fn, batched=True, batch_size=256)
test_ds = test_ds.map(tokenize_fn, batched=True, batch_size=256)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized. Example length: {len(train_ds[0]['input_ids'])}")

## 3. Fine-Tune bert-base-uncased

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1_weighted": float(f1_score(labels, preds, average="weighted")),
        "f1_macro": float(f1_score(labels, preds, average="macro")),
        "precision_weighted": float(precision_score(labels, preds, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(labels, preds, average="weighted", zero_division=0)),
    }

training_args = TrainingArguments(
    output_dir="./bert_yelp_checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}...")
start = time.perf_counter()
train_result = trainer.train()
duration = time.perf_counter() - start
print(f"\nTraining complete in {duration:.1f}s")
print(f"Final train loss: {train_result.training_loss:.4f}")

## 4. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict on test set
test_output = trainer.predict(test_ds)
test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_logits, axis=-1)
test_labels = np.array(test_df["label"].tolist())

# Metrics
test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_preds)),
    "f1_weighted": float(f1_score(test_labels, test_preds, average="weighted")),
    "f1_macro": float(f1_score(test_labels, test_preds, average="macro")),
    "precision_weighted": float(precision_score(test_labels, test_preds, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(test_labels, test_preds, average="weighted", zero_division=0)),
}

# ROC-AUC
try:
    test_metrics["roc_auc_weighted"] = float(
        roc_auc_score(test_labels, test_probs, multi_class="ovr", average="weighted")
    )
except ValueError:
    test_metrics["roc_auc_weighted"] = None

print("=" * 60)
print("TEST RESULTS: bert-base-uncased fine-tuned on Yelp")
print("=" * 60)
for k, v in test_metrics.items():
    if v is not None:
        print(f"  {k:25s}: {v:.4f}")

print(f"\nClassification Report:")
print(classification_report(
    test_labels, test_preds, target_names=LABEL_NAMES,
    labels=list(range(NUM_LABELS)), zero_division=0
))

print(f"Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

## 5. Measure Inference Latency

In [ ]:
# Latency measurement on 100 samples
sample_texts = test_df["text"].tolist()[:100]

model.eval()
times = []
for _ in range(3):
    start = time.perf_counter()
    encoded = tokenizer(
        sample_texts, max_length=MAX_LENGTH, padding=True,
        truncation=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        _ = model(**encoded)
    times.append(time.perf_counter() - start)

avg_time = np.mean(times)
latency = {
    "total_ms": round(avg_time * 1000, 2),
    "per_sample_ms": round((avg_time / 100) * 1000, 2),
    "samples_per_second": round(100 / avg_time, 1),
}

print(f"Latency (100 samples, avg of 3 runs):")
for k, v in latency.items():
    print(f"  {k}: {v}")

## 6. Save Results

In [ ]:
# Save metrics JSON
all_results = {
    "model": "bert_finetuned_bert_base_uncased_yelp",
    "base_model": MODEL_NAME,
    "dataset": "Yelp/yelp_review_full",
    "dataset_config": {
        "train_samples": len(train_split),
        "val_samples": len(val_split),
        "test_samples": len(test_df),
        "label_mapping": "5-star -> 3-class (neg/neu/pos)",
    },
    "train": {
        "train_duration_s": round(duration, 2),
        "train_loss": round(train_result.training_loss, 4),
    },
    "test": test_metrics,
    "latency": latency,
    "classification_report": classification_report(
        test_labels, test_preds, target_names=LABEL_NAMES,
        labels=list(range(NUM_LABELS)), output_dict=True, zero_division=0
    ),
    "confusion_matrix": confusion_matrix(test_labels, test_preds).tolist(),
}

with open("bert_yelp_finetuned_metrics.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Saved metrics to bert_yelp_finetuned_metrics.json")

In [ ]:
# Save model (to Google Drive or download)
SAVE_TO_DRIVE = True  # Set False to download instead

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    save_path = "/content/drive/MyDrive/models/bert_yelp_sentiment"
else:
    save_path = "./bert_yelp_sentiment"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Also save metrics alongside model
import shutil
shutil.copy("bert_yelp_finetuned_metrics.json", f"{save_path}/metrics.json")
print("Metrics copied to model directory")

In [ ]:
# Download metrics file
try:
    from google.colab import files
    files.download("bert_yelp_finetuned_metrics.json")
except ImportError:
    print("Not in Colab - file saved locally.")

## 7. Quick Comparison with Classical ML (Yelp)

| Model | F1 (weighted) | Accuracy | Latency (ms/sample) |
|-------|--------------|----------|-----------------------|
| Naive Bayes (TF-IDF) | 0.6339 | 0.6648 | 0.09 |
| Logistic Regression (TF-IDF) | 0.7473 | 0.7496 | 0.07 |
| SVM (TF-IDF) | 0.7411 | 0.7434 | 0.11 |
| **bert-base-uncased (this notebook)** | **see above** | **see above** | **see above** |